In [33]:
import pandas as pd
import os
from timezonefinderL import TimezoneFinder
from datetime import datetime
from datetime import timedelta
import pytz
from pytz import timezone

# Getting file directory
base_dir = os.getcwd()

# Loading part of first month of data for testing purposes
sw_data = pd.read_csv(base_dir+"\\..\\..\\sw_data\\2023-01-31.csv")

# Checking if data loaded properly
print(sw_data.head())

C:\Users\abrah\AppData\Local\Temp\ipykernel_5828\4109313913.py:13: DtypeWarning: Columns (12,16,17,31,39,52,108,109,110,111,112,113,114,138,139,143,146) have mixed types. Specify dtype option on import or set low_memory=False.
  sw_data = pd.read_csv(base_dir+"\\..\\..\\sw_data\\2023-01-31.csv")


                       FLT_KEY ORIG_STN_CDE DEST_STN_CDE     SCHD_ARR_CENT_TS  \
0  2023-01-01_0169_WN007_WN121        WN007        WN121  2023-01-01 09:35:00   
1  2023-01-01_2382_WN080_WN133        WN080        WN133  2023-01-01 08:15:00   
2  2023-01-01_1054_WN080_WN005        WN080        WN005  2023-01-01 08:15:00   
3  2023-01-01_2280_WN018_WN116        WN018        WN116  2023-01-01 08:35:00   
4  2023-01-01_3067_WN013_WN052        WN013        WN052  2023-01-01 10:00:00   

      SCHD_DEP_CENT_TS     ACTL_ARR_CENT_TS    ACTL_DEPT_CENT_TS  \
0  2023-01-01 05:45:00  2023-01-01 10:06:00  2023-01-01 06:17:00   
1  2023-01-01 06:45:00  2023-01-01 10:05:00  2023-01-01 07:20:00   
2  2023-01-01 06:45:00  2023-01-01 10:32:00  2023-01-01 08:45:00   
3  2023-01-01 06:45:00  2023-01-01 08:52:00  2023-01-01 07:00:00   
4  2023-01-01 07:45:00  2023-01-01 10:55:00  2023-01-01 08:37:00   

    SCHD_OFFGR_CENT_TS    SCHD_ONGR_CENT_TS   ACTL_OFFGR_CENT_TS  ...  \
0  2023-01-01 05:57:00  2023-01

In [ ]:
# ----------------------------------------------------------------
# Decoding the Origin IATA Code
# ----------------------------------------------------------------

# Dropping the duplicate Flight Keys (ignoring look_back_ts)
sw_data.drop_duplicates(subset="FLT_KEY",inplace=True)

# Finding the most frequent airport
print("Most Frequent Airport: ", sw_data["ORIG_STN_CDE"].value_counts())
print("Timezone Code Frequency: ", sw_data["ORIG_TZ_CDE"].value_counts())
print("Region Frequency: ", sw_data["ORIG_REG_SWA_CDE"].value_counts())

# Extracting data on only the most frequent airport
sw_data_c = sw_data[sw_data["ORIG_STN_CDE"] == "WN032"] # WN032 = DEN
print("Most Frequent Airport Timezone Code:", sw_data_c["ORIG_TZ_CDE"].iloc[0])

# Getting central time zone instance
central_tz = timezone("US/Central")

# Making it into actual departure time into datetime object
sw_data_c["ACTL_DEPT_CENT_TS"] = pd.to_datetime(sw_data_c["ACTL_DEPT_CENT_TS"])
sw_data_c["SCHD_DEP_CENT_TS"] = pd.to_datetime(sw_data_c["SCHD_DEP_CENT_TS"])

# Localizing to central time
sw_data_c["ACTL_DEPT_CENT_TS"] = sw_data_c["ACTL_DEPT_CENT_TS"].dt.tz_localize(central_tz)
sw_data_c["SCHD_DEP_CENT_TS"] = sw_data_c["SCHD_DEP_CENT_TS"].dt.tz_localize(central_tz)

Most Frequent Airport:  ORIG_STN_CDE
WN032    2590
WN030    2337
WN060    2268
WN073    2102
WN097    1969
         ... 
WN044       1
WN130       1
WN084       1
WN011       1
WN132       1
Name: count, Length: 129, dtype: int64
Timezone Code Frequency:  ORIG_TZ_CDE
2     13578
4     11209
1      9409
3      4094
3A     2075
6A     1048
0       288
1B      105
Name: count, dtype: int64
Region Frequency:  ORIG_REG_SWA_CDE
West          15989
Southwest      7626
Southeast      6207
Midwest        5671
East           3973
Hawaii         1048
Northwest       778
Internatio      408
PR              106
Name: count, dtype: int64
Most Frequent Airport Timezone Code: 3


C:\Users\abrah\AppData\Local\Temp\ipykernel_5828\452062956.py:21: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sw_data_c["ACTL_DEPT_CENT_TS"] = pd.to_datetime(sw_data_c["ACTL_DEPT_CENT_TS"])
C:\Users\abrah\AppData\Local\Temp\ipykernel_5828\452062956.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sw_data_c["SCHD_DEP_CENT_TS"] = pd.to_datetime(sw_data_c["SCHD_DEP_CENT_TS"])
C:\Users\abrah\AppData\Local\Temp\ipykernel_5828\452062956.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy o

In [39]:
# Crating instance of TimezoneFinder (using light version)
tf = TimezoneFinder()

# Getting central time zone instance
central_tz = timezone("US/Central")

# Loading Transtats data 156 (first month of 2023)
tnst_data = pd.read_csv(base_dir+"\\..\\..\\sw_data\\T_ONTIME_REPORTING_2023_1.csv",
                        dtype={"DEP_TIME": str, "CRS_DEP_TIME": str})

# We predict the most common airport is the DEN airport
tnst_data = tnst_data[tnst_data["ORIGIN"] == "DEN"]

# Extract only Southwest data
# tnst_data = tnst_data[tnst_data["OP_CARRIER_AIRLINE_ID"] == 19393]

# Load IATA to ICAO/Latitude/Longitude mapping
iata_icao = pd.read_csv(base_dir+"\\iata-icao.csv")
iata_to_icao = dict(zip(iata_icao["iata"], iata_icao["icao"]))
iata_to_latitude = dict(zip(iata_icao["iata"], iata_icao["latitude"]))
iata_to_longitude = dict(zip(iata_icao["iata"], iata_icao["longitude"]))

# Adding ICAO/Latitude/Longitude columns using maps
tnst_data["ICAO"] = tnst_data["ORIGIN"].map(iata_to_icao)
tnst_data["LATITUDE"] = tnst_data["ORIGIN"].map(iata_to_latitude)
tnst_data["LONGITUDE"] = tnst_data["ORIGIN"].map(iata_to_longitude)

# Adding timezone
tnst_data["LOCAL_TIMEZONE"] = tnst_data.apply(
    lambda row: tf.timezone_at(lat=row["LATITUDE"], lng=row["LONGITUDE"]),
    axis=1
)

# Converting FL_DATE column to datetime & dropping NaN's
tnst_data["FL_DATE"] = pd.to_datetime(tnst_data["FL_DATE"], format="%m/%d/%Y %I:%M:%S %p")
tnst_data.dropna(inplace=True)

# Making actual departure time date column
tnst_data["ACTL_DEP_TIME_DATE"] = (
    tnst_data["FL_DATE"] +
    pd.to_timedelta(tnst_data["DEP_TIME"].str[:2].astype(int), unit="h") +
    pd.to_timedelta(tnst_data["DEP_TIME"].str[2:].astype(int), unit="m")
)
tnst_data["SCHD_DEP_TIME_DATE"] = (
    tnst_data["FL_DATE"] +
    pd.to_timedelta(tnst_data["CRS_DEP_TIME"].str[:2].astype(int), unit="h") +
    pd.to_timedelta(tnst_data["CRS_DEP_TIME"].str[2:].astype(int), unit="m")
)

# Adding timezone awareness
tnst_data["ACTL_DEP_TIME_DATE"] = tnst_data.apply(
    lambda row: timezone(row["LOCAL_TIMEZONE"]).localize(row["ACTL_DEP_TIME_DATE"]),
    axis=1
)
tnst_data["SCHD_DEP_TIME_DATE"] = tnst_data.apply(
    lambda row: timezone(row["LOCAL_TIMEZONE"]).localize(row["SCHD_DEP_TIME_DATE"]),
    axis=1
)

# Converting to Central Time
tnst_data["ACTL_DEP_TIME_DATE_CENT"] = tnst_data["ACTL_DEP_TIME_DATE"].apply(lambda x: x.astimezone(central_tz))
tnst_data["SCHD_DEP_TIME_DATE_CENT"] = tnst_data["SCHD_DEP_TIME_DATE"].apply(lambda x: x.astimezone(central_tz))


# Removing NaN's and reseting index
tnst_data.dropna(inplace=True)
tnst_data.reset_index(inplace=True)

# Showing results
print(tnst_data.head())

   index    FL_DATE  OP_CARRIER_AIRLINE_ID TAIL_NUM  OP_CARRIER_FL_NUM ORIGIN  \
0     16 2023-01-01                  19393   N1806U               1622    DEN   
1     41 2023-01-01                  19393   N1810U                813    DEN   
2     68 2023-01-01                  19393   N207WN               3247    DEN   
3     78 2023-01-01                  19393   N210WN               2304    DEN   
4     89 2023-01-01                  19393   N211WN               3402    DEN   

  DEST CRS_DEP_TIME DEP_TIME  ICAO  LATITUDE  LONGITUDE  LOCAL_TIMEZONE  \
0  SFO         2200     2257  KDEN   39.8617   -104.673  America/Denver   
1  TUS         1440     1523  KDEN   39.8617   -104.673  America/Denver   
2  MCO         1520     1550  KDEN   39.8617   -104.673  America/Denver   
3  MCI         1350     1441  KDEN   39.8617   -104.673  America/Denver   
4  SNA         0730     0725  KDEN   39.8617   -104.673  America/Denver   

         ACTL_DEP_TIME_DATE        SCHD_DEP_TIME_DATE  \
0 202

In [42]:
# Function to find exact match in tnst_data and remove match from data
def find_match(data, sched_dt, actl_dt):  # now takes both scheduled and actual times
    for i, row in data.iterrows():
        if row["SCHD_DEP_TIME_DATE_CENT"] == sched_dt and row["ACTL_DEP_TIME_DATE_CENT"] == actl_dt:
            print("Match found: Sched =", sched_dt, "Actl =", actl_dt)
            data.drop(i, inplace=True)  # drop in-place to actually remove row
            return 1
    return 0

# Function to find matches for sw_data
def match_data(t_data, s_data):  # t_data - trnst_data, s_data - sw_data
    cnt = 0
    for i, row in s_data.iterrows():
        cnt += find_match(t_data, row["SCHD_DEP_CENT_TS"], row["ACTL_DEPT_CENT_TS"])
    print(cnt, "/", len(s_data))

# Copying datasets
tnst = tnst_data.copy()
sw   = sw_data_c.copy()

# Save both datasets to CSV
tnst.to_csv("tnst_output.csv", index=False)
sw.to_csv("sw_output.csv", index=False)

print("Files saved: tnst_output.csv and sw_output.csv")

# Running
match_data(tnst, sw)

Files saved: tnst_output.csv and sw_output.csv
Match found: Sched = 2023-01-01 08:45:00-06:00 Actl = 2023-01-01 09:09:00-06:00
Match found: Sched = 2023-01-01 08:45:00-06:00 Actl = 2023-01-01 09:59:00-06:00
Match found: Sched = 2023-01-01 13:45:00-06:00 Actl = 2023-01-01 14:27:00-06:00
Match found: Sched = 2023-01-01 12:45:00-06:00 Actl = 2023-01-01 14:11:00-06:00
Match found: Sched = 2023-01-01 14:45:00-06:00 Actl = 2023-01-01 15:39:00-06:00
Match found: Sched = 2023-01-01 15:45:00-06:00 Actl = 2023-01-01 16:07:00-06:00
Match found: Sched = 2023-01-01 15:45:00-06:00 Actl = 2023-01-01 16:12:00-06:00
Match found: Sched = 2023-01-01 14:45:00-06:00 Actl = 2023-01-01 16:57:00-06:00
Match found: Sched = 2023-01-01 13:45:00-06:00 Actl = 2023-01-01 18:52:00-06:00
Match found: Sched = 2023-01-01 17:45:00-06:00 Actl = 2023-01-01 18:18:00-06:00
Match found: Sched = 2023-01-01 17:45:00-06:00 Actl = 2023-01-01 18:12:00-06:00
Match found: Sched = 2023-01-01 17:45:00-06:00 Actl = 2023-01-01 18:49:00

In [49]:
# List to store encoded/decoded pairs
decoded_pairs = []

decoded_pairs.append({
                "IATA_ENCODED": "WN032",
                "IATA_DECODED": "DEN"
            })

# Function to find exact match in tnst_data and remove match from data
def find_match(data, sched_dt, actl_dt, iata_dest_enc):  # now takes both scheduled and actual times
    for i, row in data.iterrows():
        if row["SCHD_DEP_TIME_DATE_CENT"] == sched_dt and row["ACTL_DEP_TIME_DATE_CENT"] == actl_dt:
            decoded_iata = row["DEST"]
            print("Match found: Sched =", sched_dt, "Actl =", actl_dt)
            print("IATA_ENCODED:", iata_dest_enc, "IATA_DECODED:", decoded_iata)
            decoded_pairs.append({
                "IATA_ENCODED": iata_dest_enc,
                "IATA_DECODED": decoded_iata
            })
            data.drop(i, inplace=True)  # drop in-place to actually remove row
            return 1
    return 0

# Function to find matches for sw_data
def match_data(t_data, s_data):  # t_data - trnst_data, s_data - sw_data
    cnt = 0
    for i, row in s_data.iterrows():
        cnt += find_match(t_data, row["SCHD_DEP_CENT_TS"], row["ACTL_DEPT_CENT_TS"], row["DEST_STN_CDE"])
        if cnt > 100:
            break
    print(cnt, "/", len(s_data))

# Copying datasets
tnst = tnst_data.copy()
sw   = sw_data_c.copy()

# Save both datasets to CSV
tnst.to_csv("tnst_output.csv", index=False)
sw.to_csv("sw_output.csv", index=False)

# Running
match_data(tnst, sw)

# Saved encoded-decoded IATA mapping
decoded_df = pd.DataFrame(decoded_pairs)
decoded_df.drop_duplicates(inplace=True)
decoded_df.to_csv("decoded_iata_mapping.csv", index=False)

print("Files saved: tnst_output.csv, sw_output.csv, and decoded_iata_mapping.csv")


Match found: Sched = 2023-01-01 08:45:00-06:00 Actl = 2023-01-01 09:09:00-06:00
IATA_ENCODED: WN073 IATA_DECODED: MDW
Match found: Sched = 2023-01-01 08:45:00-06:00 Actl = 2023-01-01 09:59:00-06:00
IATA_ENCODED: WN089 IATA_DECODED: OMA
Match found: Sched = 2023-01-01 13:45:00-06:00 Actl = 2023-01-01 14:27:00-06:00
IATA_ENCODED: WN049 IATA_DECODED: HOU
Match found: Sched = 2023-01-01 12:45:00-06:00 Actl = 2023-01-01 14:11:00-06:00
IATA_ENCODED: WN053 IATA_DECODED: ICT
Match found: Sched = 2023-01-01 14:45:00-06:00 Actl = 2023-01-01 15:39:00-06:00
IATA_ENCODED: WN021 IATA_DECODED: CHS
Match found: Sched = 2023-01-01 15:45:00-06:00 Actl = 2023-01-01 16:07:00-06:00
IATA_ENCODED: WN047 IATA_DECODED: HDN
Match found: Sched = 2023-01-01 15:45:00-06:00 Actl = 2023-01-01 16:12:00-06:00
IATA_ENCODED: WN064 IATA_DECODED: LGB
Match found: Sched = 2023-01-01 14:45:00-06:00 Actl = 2023-01-01 16:57:00-06:00
IATA_ENCODED: WN063 IATA_DECODED: LGA
Match found: Sched = 2023-01-01 13:45:00-06:00 Actl = 20